# Huấn luyện model riêng (xcs-me) trên Kaggle

Dùng khi Colab không cho GPU. Kaggle miễn phí khoảng 30 giờ GPU mỗi tuần.

1. Bên phải: **Settings → Accelerator → GPU T4 x2**, và **Internet → On** (lần đầu Kaggle bắt xác minh số điện thoại).
2. Bên phải: **Input → Upload → New Dataset**, kéo file `xcs-train.jsonl` vào, đặt tên bất kỳ, bấm Create.
3. Bấm **Run All** (hoặc Save Version → Save & Run All để chạy ngầm, tắt máy cũng được).
4. Chạy xong: file `xcs-me.gguf` nằm ở **Output** (/kaggle/working) bên phải, bấm ⋮ → Download.


## 1. Cài thư viện (2–4 phút)

In [ ]:
# Hiện tiến trình cài (vài GB: torch, unsloth...). Thấy chữ chạy là đang cài, không phải bị treo.
!pip install unsloth 2>&1 | grep -E "Collecting|Downloading|Successfully|ERROR" | cut -c1-120


## 2. Tải lên dữ liệu `xcs-train.jsonl`

In [ ]:
import json, random, glob
# Lấy file .jsonl trong dataset đã thêm ở Input (bên phải)
found = sorted(glob.glob("/kaggle/input/**/*.jsonl", recursive=True))
assert found, "Chưa thấy file .jsonl: Input → Upload → New Dataset, thêm xcs-train.jsonl"
name = found[0]
print("Dùng file:", name)
rows = [json.loads(l) for l in open(name, encoding="utf-8") if l.strip()]
# Bỏ câu trùng và câu quá ngắn (1 từ) cho model học giọng tốt hơn
seen, data = set(), []
for r in rows:
    a = r["messages"][-1]["content"].strip()
    if len(a.split()) < 2 or a.lower() in seen:
        continue
    seen.add(a.lower())
    data.append(r)
random.seed(0); random.shuffle(data)
print(f"{len(rows)} dòng, dùng {len(data)} dòng")
print(data[0])

## 3. Chọn model gốc

- `unsloth/gemma-3-4b-it` (mặc định): tiếng Việt tốt, chạy trên máy bạn ~3–5 giây/câu.
- `unsloth/gemma-3-1b-it`: nằm gọn trong GPU 4 GB của bạn nên rất nhanh (~1 giây/câu), nhưng tiếng Việt kém hơn.

In [ ]:
BASE = "unsloth/gemma-3-4b-it"   # đổi thành "unsloth/gemma-3-1b-it" nếu muốn nhanh hơn
EPOCHS = 3                       # dữ liệu < 300 câu: 3; > 1000 câu: 2

from unsloth import FastModel
model, tokenizer = FastModel.from_pretrained(BASE, max_seq_length=1024, load_in_4bit=True)
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers=False, finetune_language_layers=True,
    finetune_attention_modules=True, finetune_mlp_modules=True,
    r=16, lora_alpha=16, lora_dropout=0, bias="none", random_state=3407,
)

## 4. Huấn luyện (5–15 phút)

In [ ]:
from unsloth.chat_templates import get_chat_template, train_on_responses_only
from datasets import Dataset
from trl import SFTTrainer, SFTConfig

tokenizer = get_chat_template(tokenizer, chat_template="gemma-3")
texts = [tokenizer.apply_chat_template(r["messages"], tokenize=False).removeprefix("<bos>") for r in data]
ds = Dataset.from_dict({"text": texts})

trainer = SFTTrainer(
    model=model, tokenizer=tokenizer, train_dataset=ds,
    args=SFTConfig(
        dataset_text_field="text", per_device_train_batch_size=4, gradient_accumulation_steps=2,
        num_train_epochs=EPOCHS, learning_rate=2e-4, warmup_steps=5, logging_steps=5,
        optim="adamw_8bit", weight_decay=0.01, lr_scheduler_type="linear", seed=3407,
        output_dir="out", report_to="none",
    ),
)
# Chỉ học phần câu trả lời (giọng của bạn), không học thuộc bài đăng
trainer = train_on_responses_only(trainer, instruction_part="<start_of_turn>user\n", response_part="<start_of_turn>model\n")
trainer.train()

## 5. Thử nhanh

In [ ]:
FastModel.for_inference(model)

def hoi(noi_dung, so_token=48):
    msgs = [{"role": "user", "content": noi_dung}]
    # Với gemma 3, "tokenizer" là processor: apply_chat_template trả về chữ, nên tự tách token sau
    prompt = tokenizer.apply_chat_template(msgs, add_generation_prompt=True, tokenize=False).removeprefix("<bos>")
    ids = tokenizer(text=[prompt], return_tensors="pt").to("cuda")
    for _ in range(3):
        out = model.generate(**ids, max_new_tokens=so_token, temperature=1.0, top_p=0.95, do_sample=True)
        print("-", tokenizer.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True).strip())

def thu(bai, tac_gia="ai đó"):
    hoi(f"Viết 1 bình luận trả lời bài X này theo giọng của tôi.\nTác giả: {tac_gia}\nBài: {bai}")

def bai_dang(y_mo_dau):
    # kiểu bài đăng học từ page Facebook (có khi dữ liệu có bài đã duyệt vào train)
    hoi(f"Viết bài đăng mạng xã hội theo phong cách page viết hay.\nÝ mở đầu: {y_mo_dau}", 400)

thu("Sáng nay đi biển Vũng Tàu, nắng đẹp quá trời")
thu("Mong rằng hôm nay ai cũng gặp chuyện vui, nhận được thật nhiều tin tốt lành")
bai_dang("Người trưởng thành là người biết im lặng đúng lúc")

## 6. Xuất file GGUF cho Ollama và tải về (10–15 phút)

In [ ]:
import glob, os
model.save_pretrained_gguf("xcs-gguf", tokenizer, quantization_method="Q4_K_M")
gg = [f for f in glob.glob("xcs-gguf*/**/*.gguf", recursive=True) + glob.glob("*.gguf") if "mmproj" not in f.lower()]
print(gg)
src = max(gg, key=os.path.getsize)
os.rename(src, "/kaggle/working/xcs-me.gguf")
print("Xong. Bấm link bên dưới để tải (nút Download ở mục Output hay không chạy với file lớn):")
from IPython.display import FileLink, display
os.chdir("/kaggle/working")
display(FileLink("xcs-me.gguf"))